In [1]:
import pandas as pd
import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
    
# ==========================================
# CẤU HÌNH & THAM SỐ
# ==========================================
DATA_PATH = '/kaggle/input/datasets/itachi9604/disease-symptom-description-dataset/dataset.csv'
SELECTED_FEATURES = [
    'chills', 'fatigue', 'cough', 'high_fever', 'breathlessness', 
    'phlegm', 'chest_pain', 'fast_heart_rate', 'rusty_sputum', 'malaise'
]

def log_step(message):
    print(f" ➜ {message}")

def print_header(title):
    print("\n" + "="*50)
    print(f"{title.center(50)}")
    print("="*50)

# ==========================================
# 1. TIỀN XỬ LÝ DỮ LIỆU
# ==========================================
print_header("1. PRE-PROCESSING DATA")
df = pd.read_csv(DATA_PATH)
log_step(f"Đã tải dataset: {len(df)} bản ghi.")

# Vector hóa đặc trưng
X_refined = pd.DataFrame(0, index=df.index, columns=SELECTED_FEATURES)
for col in df.columns[1:]:
    symptoms_in_col = df[col].astype(str).str.strip()
    for feat in SELECTED_FEATURES:
        X_refined.loc[symptoms_in_col == feat, feat] = 1

# Gán nhãn nhị phân
y_refined = (df['Disease'].str.strip() == 'Pneumonia').astype(int)
log_step("Đã hoàn tất trích xuất 10 đặc trưng lâm sàng.")

# Chia tập dữ liệu
X_train, X_test, y_train, y_test = train_test_split(
    X_refined, y_refined, test_size=0.2, random_state=42, stratify=y_refined
)
log_step(f"Tỉ lệ phân tách: Train ({len(X_train)}) | Test ({len(X_test)})")

# ==========================================
# 2. HUẤN LUYỆN MÔ HÌNH
# ==========================================
print_header("2. TRAINING CLINICAL MODEL")
log_step("Thuật toán: Random Forest Classifier")
log_step("Cấu hình: n_estimators=100, max_depth=5, balanced_weight")

rf_clinical = RandomForestClassifier(
    n_estimators=100, 
    max_depth=5, 
    class_weight='balanced', 
    random_state=42
)
rf_clinical.fit(X_train, y_train)
log_step("Trạng thái: Huấn luyện hoàn tất thành công.")

# ==========================================
# 3. BÁO CÁO KẾT QUẢ
# ==========================================
print_header("3. EVALUATION METRICS")
y_pred = rf_clinical.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f" [ACCURACY SCORE]: {acc:.2%}")
print("\n Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Other', 'Pneumonia']))

# Lưu trữ
joblib.dump(rf_clinical, 'symptom_model_refined.pkl')
joblib.dump(SELECTED_FEATURES, 'symptoms_list_refined.pkl')
log_step("Đã lưu Model & Features List thành công!")

# ==========================================
# 4. KIỂM THỬ THỰC TẾ (INFERENCE)
# ==========================================
print_header("4. MODEL INFERENCE TEST")

def predict_test(symptoms):
    input_v = pd.DataFrame(0, index=[0], columns=SELECTED_FEATURES)
    for s in symptoms:
        if s in SELECTED_FEATURES: input_v.at[0, s] = 1
    prob = rf_clinical.predict_proba(input_v)[0][1]
    
    print(f" ▸ Input Symptoms: {', '.join(symptoms)}")
    print(f" ▸ Result: {prob*100:.2f}% xác suất mắc Viêm phổi")

# Chạy test
predict_test(['cough', 'high_fever', 'rusty_sputum'])
print("="*50 + "\n")


              1. PRE-PROCESSING DATA              
 ➜ Đã tải dataset: 4920 bản ghi.
 ➜ Đã hoàn tất trích xuất 10 đặc trưng lâm sàng.
 ➜ Tỉ lệ phân tách: Train (3936) | Test (984)

            2. TRAINING CLINICAL MODEL            
 ➜ Thuật toán: Random Forest Classifier
 ➜ Cấu hình: n_estimators=100, max_depth=5, balanced_weight
 ➜ Trạng thái: Huấn luyện hoàn tất thành công.

              3. EVALUATION METRICS               
 [ACCURACY SCORE]: 100.00%

 Detailed Classification Report:
              precision    recall  f1-score   support

       Other       1.00      1.00      1.00       960
   Pneumonia       1.00      1.00      1.00        24

    accuracy                           1.00       984
   macro avg       1.00      1.00      1.00       984
weighted avg       1.00      1.00      1.00       984

 ➜ Đã lưu Model & Features List thành công!

             4. MODEL INFERENCE TEST              
 ▸ Input Symptoms: cough, high_fever, rusty_sputum
 ▸ Result: 35.00% xác suất mắc Viê

New train

In [2]:
# -*- coding: utf-8 -*-
"""
HUẤN LUYỆN LẠI NHÁNH LÂM SÀNG (triệu chứng) — phiên bản trung thực.

Thay đổi so với bản cũ:
  - Đổi mô hình chính: RandomForest -> Logistic Regression (có regularization L2).
    Lý do: trên dataset xác định, RF học thuộc "trọn bộ triệu chứng" nên hành xử
    BRITTLE với ca không đầy đủ (vd: cho 3/10 triệu chứng -> xác suất tụt vô lý).
    LR cho xác suất ĐƠN ĐIỆU & GIẢI THÍCH ĐƯỢC theo bằng chứng triệu chứng,
    phù hợp vai trò "tầng nudge" trong fusion lấy X-quang làm trụ.
  - Đánh giá bằng Stratified K-Fold + báo cáo Precision/Recall/F1 LỚP Pneumonia
    (bỏ accuracy làm chỉ số khoe, vì lớp mất cân bằng).
  - Giữ RandomForest lại CHỈ để so sánh/ablation trong báo cáo.
"""
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report)

# ===================== CẤU HÌNH =====================
DATA_PATH = '/kaggle/input/datasets/itachi9604/disease-symptom-description-dataset/dataset.csv'
SELECTED_FEATURES = [
    'chills', 'fatigue', 'cough', 'high_fever', 'breathlessness',
    'phlegm', 'chest_pain', 'fast_heart_rate', 'rusty_sputum', 'malaise'
]
RANDOM_STATE = 42
N_SPLITS = 5

def log_step(m): print(f" \u279c {m}")
def print_header(t):
    print("\n" + "="*58); print(t.center(58)); print("="*58)

# ===================== 1. TIỀN XỬ LÝ =====================
print_header("1. TIEN XU LY DU LIEU")
df = pd.read_csv(DATA_PATH)
log_step(f"Da tai dataset: {len(df)} ban ghi.")

X = pd.DataFrame(0, index=df.index, columns=SELECTED_FEATURES)
for col in df.columns[1:]:
    s = df[col].astype(str).str.strip()
    for feat in SELECTED_FEATURES:
        X.loc[s == feat, feat] = 1
y = (df['Disease'].str.strip() == 'Pneumonia').astype(int)
log_step(f"Trich xuat {len(SELECTED_FEATURES)} dac trung. "
         f"Ti le duong (Pneumonia): {y.mean():.2%} ({int(y.sum())} ca).")

# ===================== 2. ĐÁNH GIÁ CHÉO =====================
print_header("2. DANH GIA CHEO (Stratified 5-Fold)")
models = {
    "RandomForest (cu)": RandomForestClassifier(
        n_estimators=100, max_depth=5, class_weight='balanced', random_state=RANDOM_STATE),
    "LogisticReg (moi)": LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
}
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
print(f"\n{'Model':<20}{'Acc':>8}{'Precision':>11}{'Recall':>9}{'F1':>8}   (lop Pneumonia)")
print("-"*64)
for name, model in models.items():
    yp = cross_val_predict(model, X, y, cv=cv)
    print(f"{name:<20}{accuracy_score(y,yp):>7.1%}"
          f"{precision_score(y,yp,zero_division=0):>11.1%}"
          f"{recall_score(y,yp,zero_division=0):>9.1%}"
          f"{f1_score(y,yp,zero_division=0):>8.1%}")
log_step("Dataset tach lop gan nhu hoan hao -> ca hai model deu rat cao.")
log_step("=> Con so cao phan anh tinh XAC DINH cua dataset, khong phai nang luc lam sang.")

# ===================== 3. ĐỘ GIẢI THÍCH =====================
print_header("3. DO GIAI THICH (trong so trieu chung)")
rf = models["RandomForest (cu)"].fit(X, y)
lr = models["LogisticReg (moi)"].fit(X, y)
coef, imp = lr.coef_[0], rf.feature_importances_
print(f"\n{'Trieu chung':<18}{'He so LR':>12}{'Importance RF':>16}")
print("-"*46)
for i in np.argsort(-np.abs(coef)):
    print(f"{SELECTED_FEATURES[i]:<18}{coef[i]:>+12.3f}{imp[i]:>16.3f}")
log_step("He so LR cho biet HUONG (+/-) va DO MANH cua tung trieu chung.")
log_step("Trieu chung dac hieu (vd rusty_sputum) co he so duong lon -> dac trung cho viem phoi.")

# ===================== 4. HÀNH VI TRÊN CA KHÔNG ĐẦY ĐỦ =====================
print_header("4. SO SANH HANH VI TREN CA KHONG DAY DU")
def prob(model, symptoms):
    v = pd.DataFrame(0, index=[0], columns=SELECTED_FEATURES)
    for s in symptoms:
        if s in SELECTED_FEATURES: v.at[0, s] = 1
    return model.predict_proba(v)[0][1]

order_sym = ['cough','high_fever','rusty_sputum','phlegm','chest_pain',
             'breathlessness','fast_heart_rate','chills','fatigue','malaise']
print(f"\n{'So trieu chung':<16}{'RandomForest':>14}{'LogisticReg':>14}")
print("-"*44)
for k in range(1, len(order_sym)+1):
    sub = order_sym[:k]
    print(f"{k:<16}{prob(rf,sub):>13.1%}{prob(lr,sub):>14.1%}")
log_step("RF: nhay phi don dieu, tut thap khi trieu chung chua du bo (brittle).")
log_step("LR: tang theo bang chung trieu chung -> hop ly lam sang & on cho fusion.")

classic = ['cough','high_fever','rusty_sputum']
print(f"\n Ca kinh dien {classic}:")
print(f"   RandomForest (cu): {prob(rf,classic):.1%}")
print(f"   LogisticReg (moi): {prob(lr,classic):.1%}")

# ===================== 5. LƯU MODEL =====================
print_header("5. LUU TRU")
joblib.dump(lr, 'symptom_model_lr.pkl')
joblib.dump(SELECTED_FEATURES, 'symptoms_list.pkl')
log_step("Da luu Logistic Regression lam model lam sang chinh cho tang fusion.")
log_step("RandomForest chi giu de so sanh/ablation trong bao cao.")
print("="*58)


                  1. TIEN XU LY DU LIEU                   
 ➜ Da tai dataset: 4920 ban ghi.
 ➜ Trich xuat 10 dac trung. Ti le duong (Pneumonia): 2.44% (120 ca).

           2. DANH GIA CHEO (Stratified 5-Fold)           

Model                    Acc  Precision   Recall      F1   (lop Pneumonia)
----------------------------------------------------------------
RandomForest (cu)    100.0%     100.0%   100.0%  100.0%
LogisticReg (moi)    100.0%     100.0%   100.0%  100.0%
 ➜ Dataset tach lop gan nhu hoan hao -> ca hai model deu rat cao.
 ➜ => Con so cao phan anh tinh XAC DINH cua dataset, khong phai nang luc lam sang.

         3. DO GIAI THICH (trong so trieu chung)          

Trieu chung           He so LR   Importance RF
----------------------------------------------
rusty_sputum            +5.171           0.285
fast_heart_rate         +4.244           0.305
breathlessness          +1.250           0.105
chest_pain              +0.953           0.079
chills                  +0.811   